# 🎨 Google Colab + TurboFlow Extension Integration
## Auto-generate images via local extension, create shorts locally

**Setup:**
1. ✅ Make sure TurboFlow extension is loaded (dev mode) on your local Chrome
2. ✅ Run `python bridge_local.py` on your local machine
3. 📓 Use this notebook to: create scripts → send prompts to extension → get images → render videos

---

## 🔧 Step 1: Configure Bridge Connection
Replace `YOUR_LOCAL_IP` with your machine's local IP (e.g., `192.168.1.100`)

In [ ]:
# ⚙️ CONFIGURATION
BRIDGE_HOST = "YOUR_LOCAL_IP"  # Or "127.0.0.1" if Colab is on same machine (rare)
BRIDGE_PORT = 8787
BRIDGE_URL = f"http://{BRIDGE_HOST}:{BRIDGE_PORT}"
POLL_INTERVAL = 2  # seconds, check for new images
TIMEOUT_WAIT_IMAGES = 300  # max 5 min to wait for images

# Colab paths
from pathlib import Path
import os
COLAB_ROOT = Path("/content")
PROJECTS_DIR = COLAB_ROOT / "projects"  # Will sync with local render_web.py
PROJECTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"🌉 Bridge: {BRIDGE_URL}")
print(f"📁 Projects dir: {PROJECTS_DIR}")

## 🏥 Step 2: Test Bridge Connection

In [ ]:
import requests
import json
from time import sleep

def test_bridge():
    try:
        resp = requests.get(f"{BRIDGE_URL}/health", timeout=5)
        if resp.status_code == 200:
            print("✅ Bridge is healthy!")
            return True
        else:
            print(f"❌ Bridge returned {resp.status_code}")
            return False
    except requests.exceptions.ConnectionError:
        print(f"❌ Cannot connect to {BRIDGE_URL}")
        print("   Make sure:")
        print("   1. TurboFlow extension is loaded in Chrome dev mode")
       print("   2. Run: python bridge_local.py on your local machine")
        print(f"   3. Replace BRIDGE_HOST with your machine IP (not {BRIDGE_HOST})")
        return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

test_bridge()

## 📤 Step 3: Queue Job (Send Prompts to Extension)
The extension will create images for each prompt and save them to Downloads

In [ ]:
def enqueue_job(prompts: list, job_name: str = None, naming_prefix: str = "prompt"):
    """
    Queue a batch of prompts for extension to generate images.
    
    Args:
        prompts: List of text prompts (e.g., ["a cat", "a dog", "a bird"])
        job_name: Optional job identifier (default: auto-generated)
        naming_prefix: Prefix for saved images (e.g., "prompt-001.jpg", "prompt-002.jpg")
    
    Returns:
        job_id: Unique job identifier to track progress
    """
    import uuid
    from datetime import datetime
    
    job_id = job_name or f"job_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    payload = {
        "job_id": job_id,
        "prompts": prompts,
        "settings": {
            "naming": "prefix",
            "namingPrefix": naming_prefix,
            "namingSeparator": "-",
            "imageRatio": "IMAGE_ASPECT_RATIO_LANDSCAPE",
            "imageCount": 1,
            "autoDownloadImages": True
        }
    }
    
    try:
        resp = requests.post(f"{BRIDGE_URL}/enqueue", json=payload, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            print(f"✅ Job queued: {job_id}")
            print(f"   Prompts: {len(prompts)}")
            print(f"   Images will be saved as: {naming_prefix}-001.jpg, {naming_prefix}-002.jpg, ...")
            return job_id
        else:
            print(f"❌ Failed to enqueue: HTTP {resp.status_code}")
            print(resp.text)
            return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# 📝 Example usage
# TEST_PROMPTS = [
#     "a person eating breakfast at a cafe",
#     "a close-up of coffee being poured",
#     "a sunny morning kitchen"
# ]
# job_id = enqueue_job(TEST_PROMPTS, job_name="test_breakfast", naming_prefix="breakfast")
# print(f"\n🔖 Job ID: {job_id}")

## 📊 Step 4: Monitor & Download Generated Images

In [ ]:
def wait_for_images(job_id: str, naming_prefix: str, expected_count: int, timeout_sec: int = 300):
    """
    Poll bridge for generated images and download them to local project.
    
    Args:
        job_id: Job ID returned from enqueue_job()
        naming_prefix: Same prefix used in enqueue_job()
        expected_count: Expected number of images
        timeout_sec: Max time to wait in seconds
    
    Returns:
        List of downloaded image filenames
    """
    import time
    from tqdm import tqdm
    
    print(f"\n⏳ Waiting for {expected_count} images (max {timeout_sec}s)...")
    start = time.time()
    downloaded = set()
    
    with tqdm(total=expected_count, desc="Images") as pbar:
        while len(downloaded) < expected_count:
            if time.time() - start > timeout_sec:
                print(f"⏱️ Timeout! Got {len(downloaded)}/{expected_count} images")
                break
            
            try:
                resp = requests.get(
                    f"{BRIDGE_URL}/images",
                    params={"job_id": job_id, "prefix": naming_prefix},
                    timeout=5
                )
                if resp.status_code == 200:
                    items = resp.json().get("items", [])
                    new_images = []
                    for item in items:
                        name = item["name"]
                        if name not in downloaded:
                            new_images.append(name)
                            downloaded.add(name)
                    
                    # Download new images
                    for name in new_images:
                        try:
                            img_resp = requests.get(
                                f"{BRIDGE_URL}/download",
                                params={"name": name},
                                timeout=30
                            )
                            if img_resp.status_code == 200:
                                # Save to projects/<naming_prefix>/images/
                                img_dir = PROJECTS_DIR / naming_prefix / "images"
                                img_dir.mkdir(parents=True, exist_ok=True)
                                img_path = img_dir / name
                                img_path.write_bytes(img_resp.content)
                                pbar.update(1)
                        except Exception as e:
                            print(f"   ⚠️ Failed to download {name}: {e}")
            except requests.exceptions.Timeout:
                pass  # Retry
            except Exception as e:
                print(f"   ⚠️ Poll error: {e}")
            
            if len(downloaded) < expected_count:
                time.sleep(POLL_INTERVAL)
    
    print(f"\n✅ Downloaded {len(downloaded)}/{expected_count} images")
    return sorted(list(downloaded))

# # Example:
# images = wait_for_images(
#     job_id="test_breakfast",
#     naming_prefix="breakfast",
#     expected_count=3
# )

## 🎬 Step 5: Generate Script + Queue Images + Render Video
Full pipeline: topic → script → TTS → images (via extension) → video

In [ ]:
def create_short_from_topic(
    topic: str,
    script_text: str = None,
    audio_path: str = None,
    srt_path: str = None,
    keywords_json: dict = None,
    use_extension_images: bool = True
):
    """
    Complete pipeline: topic → script → audio/srt → queue images → render video
    
    Args:
        topic: Video topic (e.g., "How to use Get in English")
        script_text: Pre-written script (optional, auto-generate if not provided)
        audio_path: Path to audio.mp3 (required)
        srt_path: Path to subtitle.srt (required)
        keywords_json: Pre-generated keywords (optional)
        use_extension_images: If True, queue images to extension; if False, use Pexels
    
    Returns:
        project_dir: Path to the generated project directory
    """
    import re
    from pathlib import Path
    from datetime import datetime
    import json
    import shutil
    
    # Step 1: Create project directory
    slug = re.sub(r"[^a-z0-9]+", "_", topic.lower()).strip("_")
    project_dir = PROJECTS_DIR / slug
    project_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"📁 Project: {project_dir}")
    
    # Step 2: Copy audio & srt
    if audio_path:
        shutil.copy(audio_path, project_dir / "audio.mp3")
        print(f"   ✅ Copied audio.mp3")
    
    if srt_path:
        shutil.copy(srt_path, project_dir / "subtitles.srt")
        print(f"   ✅ Copied subtitles.srt")
    
    # Step 3: Save script & keywords
    if script_text:
        (project_dir / "script.txt").write_text(script_text, encoding="utf-8")
        print(f"   ✅ Saved script.txt")
    
    if keywords_json:
        (project_dir / "keywords.json").write_text(
            json.dumps(keywords_json, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )
        print(f"   ✅ Saved keywords.json")
        
        # Step 4: Queue images if using extension
        if use_extension_images:
            visual_keywords = keywords_json.get("visual_keywords", [])
            if visual_keywords:
                prompts = [kw["search_query"] for kw in visual_keywords]
                job_id = enqueue_job(prompts, job_name=slug, naming_prefix=slug)
                
                if job_id:
                    print(f"\n   📤 Queued {len(prompts)} images from extension")
                    
                    # Wait for images
                    images = wait_for_images(
                        job_id=job_id,
                        naming_prefix=slug,
                        expected_count=len(prompts),
                        timeout_sec=600
                    )
                    print(f"   ✅ Downloaded {len(images)} images")
    
    print(f"\n✅ Project ready: {project_dir}")
    print(f"   Next: Use render_web.py to generate video")
    print(f"   Command: python render_web.py {slug}")
    
    return project_dir

## 📋 Example: Full Workflow
Complete example from topic to video

In [ ]:
# 📝 Option A: Manual setup (you provide audio/srt)
# Example paths (upload your audio.mp3 and subtitles.srt first)
# AUDIO_PATH = "/content/audio.mp3"
# SRT_PATH = "/content/subtitles.srt"

# SCRIPT = """This is my script text.
# It should match your audio and subtitles.
# """

# KEYWORDS = {
#     "script": SCRIPT,
#     "visual_keywords": [
#         {"keyword": "morning", "search_query": "sunrise morning light"},
#         {"keyword": "coffee", "search_query": "hot coffee cup morning"},
#         {"keyword": "desk", "search_query": "clean desk workspace"},
#     ]
# }

# project_dir = create_short_from_topic(
#     topic="Morning Routine",
#     script_text=SCRIPT,
#     audio_path=AUDIO_PATH,
#     srt_path=SRT_PATH,
#     keywords_json=KEYWORDS,
#     use_extension_images=True
# )

print("✅ Example workflow defined. Uncomment above to run.")

## 🛠️ Helper Functions

In [ ]:
def list_projects():
    """List all projects in /content/projects"""
    if PROJECTS_DIR.exists():
        projects = [p.name for p in PROJECTS_DIR.iterdir() if p.is_dir()]
        if projects:
            print(f"📁 Projects ({len(projects)}):")
            for p in sorted(projects):
                print(f"   - {p}")
        else:
            print("   (No projects yet)")
    else:
        print("Projects dir not found")

def get_job_status(job_id: str):
    """Get current job status from bridge"""
    try:
        resp = requests.get(f"{BRIDGE_URL}/status", params={"job_id": job_id}, timeout=5)
        if resp.status_code == 200:
            job = resp.json().get("job", {})
            status = job.get("status")
            stats = job.get("stats", {})
            print(f"📊 Job: {job_id}")
            print(f"   Status: {status}")
            if stats:
                print(f"   Total: {stats.get('total')}, Downloaded: {stats.get('downloaded')}, Failed: {stats.get('failed')}")
        else:
            print(f"❌ Job not found: {job_id}")
    except Exception as e:
        print(f"❌ Error: {e}")

def clear_project(project_name: str):
    """Delete a project directory"""
    import shutil
    project_dir = PROJECTS_DIR / project_name
    if project_dir.exists():
        shutil.rmtree(project_dir)
        print(f"🗑️ Deleted: {project_name}")
    else:
        print(f"❌ Project not found: {project_name}")

# Examples:
# list_projects()
# get_job_status("test_breakfast")
# clear_project("test_breakfast")

## 📌 Notes

### How it works:
1. **Local Bridge Server** (`bridge_local.py`)
   - Runs on your machine at `http://127.0.0.1:8787`
   - Extension polls `/next` endpoint for jobs
   - Bridge watches Downloads folder for new images

2. **Extension Integration** (`dev-bridge.js`)
   - Polls bridge for jobs
   - Receives prompts → sends to Google Flow
   - Flow generates images → saves to Downloads
   - Sends status updates back to bridge

3. **Colab Workflow**
   - Queue jobs via `/enqueue`
   - Poll `/images` to check progress
   - Download images via `/download`
   - Save to local project folders
   - Run `render_web.py` locally to create video

### Image Naming Convention
```
enqueue_job(
    ["a cat", "a dog"],
    naming_prefix="animals"  # ← filename prefix
)
# Creates: animals-001.jpg, animals-002.jpg, ...
```

### Troubleshooting
- **Bridge won't connect**: Check BRIDGE_HOST is correct (use local machine IP, not localhost)
- **Images not generating**: Make sure extension is running and Flow tab is open
- **Images take too long**: Extension limits are set to unlimited (dev mode), but Google Flow may throttle
- **Missing images**: Check Downloads folder on local machine for errors